In [10]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
import csep
from csep.core.catalogs import CSEPCatalog
from csep.utils.time_utils import datetime_to_utc_epoch

In [11]:
catalog_dir = Path("/Users/rahulravi23/Desktop/Work/seismic_hazard_modelling/seismic_hazard_modelling/data/Earthquake_Catalog/")
output_dir = "/Users/rahulravi23/Desktop/Work/seismic_hazard_modelling/seismic_hazard_modelling/data/processed_catalogs/"
# os.mkdir(output_dir)

In [12]:
MC_FINAL = {
    "Kanto_Japan":                  4.4,
    "Tohoku_Japan":                 4.4,
    "Central_Chile":                4.0,
    "Central_Turkey":               4.3,
    "Central_Nepal":                4.4,
    "North_Island_NZ":              4.0,
    "Sichuan_China":                4.3,
    "Southern_Sumatra_Indonesia":   4.5,
    "Western_Australia":            3.0,
    "Kuch_India":                   4.0,
    "Ordos_China":                  4.0,
    "Southern_Norway":              2.5,
}

def gk_windows(magnitude):
    time_days = 10 ** (0.5 * magnitude - 0.8)
    dist_km   = 10 ** (0.1238 * magnitude + 0.983)
    return time_days, dist_km

def haversine_km(lat1, lon1, lat2, lon2):
    """Vectorised haversine distance in km."""
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

def gardner_knopoff(df):
    """
    Returns boolean array: True = mainshock, False = aftershock/foreshock.
    Works on a DataFrame with columns: time (datetime), latitude, longitude, mag.
    Sorted by time ascending assumed.
    """
    df = df.sort_values('time').reset_index(drop=True)
    n = len(df)
    is_mainshock = np.ones(n, dtype=bool)  # start: all mainshocks

    lats = df['latitude'].values
    lons = df['longitude'].values
    mags = df['mag'].values
    times = df['time'].values.astype('datetime64[s]').astype(np.int64)  # seconds

    for i in range(n):
        if not is_mainshock[i]:
            continue  # already classified as aftershock — skip

        t_win_days, d_win_km = gk_windows(mags[i])
        t_win_sec = t_win_days * 86400

        # Look forward in time only
        for j in range(i + 1, n):
            dt_sec = float(times[j] - times[i])
            if dt_sec > t_win_sec:
                break  # sorted by time — no need to look further
            dist = haversine_km(lats[i], lons[i], lats[j], lons[j])
            if dist <= d_win_km:
                if mags[j] <= mags[i]:
                    is_mainshock[j] = False  # j is aftershock of i

    return is_mainshock

In [13]:
results = []

for f in sorted(catalog_dir.glob("*.csv")):
    patch = f.stem
    mc = MC_FINAL.get(patch)
    if mc is None:
        print(f"{patch} — no Mc defined, skipping")
        continue

    df = pd.read_csv(f)
    df['time'] = pd.to_datetime(df['time'], utc=True)
    df_trimmed = df[df['mag'] >= mc].copy().reset_index(drop=True)

    if len(df_trimmed) < 10:
        print(f"{patch:<35} SKIPPED — {len(df_trimmed)} events above Mc={mc}")
        df_trimmed.to_csv(output_dir + f"{patch}_full.csv", index=False)
        results.append({
            'patch': patch, 'mc': mc,
            'n_raw': len(df),
            'n_above_mc': len(df_trimmed),
            'n_mainshocks': len(df_trimmed),
            'n_aftershocks': 0,
            'aftershock_pct': 0.0,
            'status': 'too_sparse'
        })
        continue

    print(f"Processing {patch} ({len(df_trimmed)} events above Mc={mc})...")

    is_main = gardner_knopoff(df_trimmed)
    df_mainshocks  = df_trimmed[is_main].reset_index(drop=True)
    df_aftershocks = df_trimmed[~is_main].reset_index(drop=True)
    aftershock_pct = 100 * len(df_aftershocks) / len(df_trimmed)

    print(f"  Mainshocks:  {len(df_mainshocks):>5} ({100-aftershock_pct:.1f}%)")
    print(f"  Aftershocks: {len(df_aftershocks):>5} ({aftershock_pct:.1f}%)")

    # Save both versions
    df_trimmed.to_csv(    output_dir + f"{patch}_full.csv",        index=False)
    df_mainshocks.to_csv( output_dir + f"{patch}_declustered.csv", index=False)

    results.append({
        'patch': patch, 'mc': mc,
        'n_raw': len(df),
        'n_above_mc': len(df_trimmed),
        'n_mainshocks': len(df_mainshocks),
        'n_aftershocks': len(df_aftershocks),
        'aftershock_pct': round(aftershock_pct, 1),
        'status': 'ok'
    })

Processing Central_Chile (1333 events above Mc=4.0)...
  Mainshocks:    530 (39.8%)
  Aftershocks:   803 (60.2%)
Processing Central_Nepal (200 events above Mc=4.4)...
  Mainshocks:     49 (24.5%)
  Aftershocks:   151 (75.5%)
Processing Central_Turkey (442 events above Mc=4.3)...
  Mainshocks:    108 (24.4%)
  Aftershocks:   334 (75.6%)
Processing Kanto_Japan (2061 events above Mc=4.4)...
  Mainshocks:    773 (37.5%)
  Aftershocks:  1288 (62.5%)
Processing Kuch_India (28 events above Mc=4.0)...
  Mainshocks:     24 (85.7%)
  Aftershocks:     4 (14.3%)
Processing North_Island_NZ (1014 events above Mc=4.0)...
  Mainshocks:    760 (75.0%)
  Aftershocks:   254 (25.0%)
Processing Ordos_China (27 events above Mc=4.0)...
  Mainshocks:     27 (100.0%)
  Aftershocks:     0 (0.0%)
Processing Sichuan_China (679 events above Mc=4.3)...
  Mainshocks:    148 (21.8%)
  Aftershocks:   531 (78.2%)
Southern_Norway                     SKIPPED — 4 events above Mc=2.5
Processing Southern_Sumatra_Indonesia (

In [14]:
# Summary
results_df = pd.DataFrame(results)
print("\n" + "="*85)
print("DECLUSTERING SUMMARY (Gardner-Knopoff)")
print("="*85)
print(results_df.to_string(index=False))
results_df.to_csv("/Users/rahulravi23/Desktop/Work/seismic_hazard_modelling/seismic_hazard_modelling/data/declustering_summary.csv", index=False)


DECLUSTERING SUMMARY (Gardner-Knopoff)
                     patch  mc  n_raw  n_above_mc  n_mainshocks  n_aftershocks  aftershock_pct     status
             Central_Chile 4.0   2597        1333           530            803            60.2         ok
             Central_Nepal 4.4    409         200            49            151            75.5         ok
            Central_Turkey 4.3    750         442           108            334            75.6         ok
               Kanto_Japan 4.4   3149        2061           773           1288            62.5         ok
                Kuch_India 4.0     32          28            24              4            14.3         ok
           North_Island_NZ 4.0   1470        1014           760            254            25.0         ok
               Ordos_China 4.0     30          27            27              0             0.0         ok
             Sichuan_China 4.3   1154         679           148            531            78.2         ok
      

## Insights

### Overview

<p>Declustering was performed as the second catalog processing step, following Mc trimming. The goal is to produce two distinct catalog versions for each patch serving different roles in the modelling pipeline. The full sequence catalog retains all events above Mc including aftershocks and foreshocks. This is the input to the temporal GNN, which must learn to encode aftershock sequences, Omori decay patterns, and rate changes following mainshocks as meaningful temporal signals. The declustered mainshock-only catalog removes clustered events, leaving only the background Poissonian seismicity — this is used for computing static background rates, b-value estimates, and any features that assume temporal independence between events.</p>

### Method

<p>The Gardner-Knopoff (1974) declustering algorithm was implemented from scratch rather than relying on the pyCSEP library, whose internal API had changed between versions making direct use unreliable. The Gardner-Knopoff method defines magnitude-dependent space-time windows around each event: any subsequent event occurring within the time window (T = 10^(0.5M - 0.8) days) and distance window (D = 10^(0.1238M + 0.983) km) of a larger event is classified as an aftershock and removed. The algorithm processes events in chronological order, treating each unclassified event as a potential mainshock and flagging subsequent events within its space-time window as aftershocks. The haversine formula was used for all distance calculations to correctly account for the curvature of the Earth across the spatial scales involved.</p>

<p>Both catalog versions, full sequence and declustered, were saved for every patch and will be used at distinct stages of the feature engineering pipeline. No patch's declustered catalog was used where the full sequence catalog was intended, and vice versa. This separation is maintained strictly throughout all subsequent processing steps.</p>

### Results & Physical Interpretation

<p>Aftershock fractions vary substantially across patches and are directly interpretable in terms of the major seismic sequences that occurred within each patch during the 2000–2024 study period. The five patches that experienced Mw ≥ 7.5 mainshocks during this period — Tohoku (2011 Mw 9.1), Sichuan (2008 Mw 7.9), Nepal (2015 Mw 7.8), Turkey (2023 Mw 7.8/7.5), and Chile (2010 Mw 8.8 + 2015 Mw 8.3) — all show aftershock fractions above 60%, with Tohoku reaching 86.4%. These high fractions are physically expected and scientifically meaningful: they reflect the dominance of a single catastrophic sequence over the background seismicity rate, and the temporal GNN is specifically designed to learn the dynamics of such sequences as transferable patterns.</p>

<p>The contrast between high-aftershock and low-aftershock patches reveals a second important signal. New Zealand (25.0%), Western Australia (19.4%), and Kutch (14.3%) show low aftershock fractions despite varying seismicity levels — NZ because its high catalog completeness captures many independent background events that don't cluster tightly around mainshocks, Australia and Kutch because their intraplate settings produce rare isolated events without sustained aftershock sequences. Ordos (0.0%) confirms stable craton background seismicity with no detectable clustering among its 27 events above Mc. This range of aftershock fractions — from 0% to 86% — spans the full spectrum of tectonic behaviour and provides the temporal GNN with a rich and diverse set of sequence types to learn from during global pre-training.</p>

#### Flag: Nepal mainshock catalog

<p>After declustering, Nepal retains only 49 mainshocks above Mc=4.4 — approximately 2 independent events per year over the 24-year study period. This is the thinnest mainshock catalog of any patch with sufficient events for modelling and creates a practical constraint on temporal feature engineering: rolling window statistics computed on the declustered catalog will be sparse or NaN for most time steps. For Nepal, the full sequence catalog (200 events) will be the primary input to the temporal GNN, and declustered catalog features will be masked where event counts fall below the minimum threshold for reliable estimation. This is documented as a known limitation of the Nepal patch and will be discussed in the paper alongside the broader challenge of data-sparse high-seismicity regions where large mainshocks dominate the observable catalog.</p>